# Programming Languages: from regular expressions to a virtual machine

**Software Development & Python — 90-minute seminar**

Today we will build a tiny language end to end, using only strings, numbers,
tuples, lists, dictionaries, and functions.

```text
Source → Lexer → Tokens → Parser → AST → Interpreter
                                      └→ Compiler → Bytecode → VM
```


## 1. Regular expressions

A regular expression describes a text pattern. Literal characters match
themselves; character classes such as `[a-z]` choose from a set; `|` means an
alternative; `*`, `+`, and `?` mean zero-or-more, one-or-more, and optional;
parentheses group a sub-pattern.

Useful patterns today:

- `r"\d+"` — one or more digits
- `r"[a-z]+"` — one or more lowercase letters
- `r"\d+(\.\d+)?"` — an integer with an optional decimal part
- `r"[+\-*/()]"` — one arithmetic symbol


In [2]:
import re

print(re.findall(r"\d+", "12 + 34 * 5"))
print(re.findall(r"[+\-*/()]", "(12 + 3) / 5"))
print(re.fullmatch(r"\d+", "123"))
print(re.fullmatch(r"\d+", "12x"))
print(re.fullmatch(r"\d+(\.\d+)?", "12.50"))


['12', '34', '5']
['(', '+', ')', '/']
<re.Match object; span=(0, 3), match='123'>
None
<re.Match object; span=(0, 5), match='12.50'>


### Regular expressions and finite automata

For **classical** regular expressions, these three views describe the same
kind of pattern:

```text
classical regular expressions ↔ regular languages ↔ finite automata

              digit
start ─────────────→ NUMBER
                       ↑  │
                       └──┘ digit
```

The pattern `[0-9]+` starts with a digit and stays in the `NUMBER` state while
digits continue. A lexer can be understood as moving through finitely many
states while scanning input. Python's `re` also has features beyond classical
regular expressions, so the equivalence above refers specifically to the
classical form.


## 2. Lexing arithmetic expressions

Our initial language has integer literals, `+`, `-`, `*`, `/`, and
parentheses. It deliberately has no names, assignment, strings, unary minus,
or control flow. The lexer turns characters into token tuples.


In [3]:
TOKEN_PATTERN = re.compile(
    r"(?P<NUMBER>\d+)"
    r"|(?P<PLUS>\+)"
    r"|(?P<MINUS>-)"
    r"|(?P<MUL>\*)"
    r"|(?P<DIV>/)"
    r"|(?P<LPAREN>\()"
    r"|(?P<RPAREN>\))"
    r"|(?P<SPACE>\s+)"
    r"|(?P<MISMATCH>.)"
)

def tokenize(source):
    tokens = []
    for match in TOKEN_PATTERN.finditer(source):
        kind = match.lastgroup
        text = match.group()
        if kind == "SPACE":
            continue
        if kind == "MISMATCH":
            raise ValueError("Unexpected character: " + text)
        tokens.append((kind, text))
    return tokens


In [4]:
expected_tokens = [
    ("NUMBER", "12"), ("PLUS", "+"), ("NUMBER", "3"),
    ("MUL", "*"), ("LPAREN", "("), ("NUMBER", "4"),
    ("MINUS", "-"), ("NUMBER", "1"), ("RPAREN", ")"),
]
actual_tokens = tokenize("12 + 3 * (4 - 1)")
print(actual_tokens)
assert actual_tokens == expected_tokens


[('NUMBER', '12'), ('PLUS', '+'), ('NUMBER', '3'), ('MUL', '*'), ('LPAREN', '('), ('NUMBER', '4'), ('MINUS', '-'), ('NUMBER', '1'), ('RPAREN', ')')]


## 3. Parsing and AST

The lexer recognizes local pieces; the parser recognizes structure. For
`1 + 2 * 3`, multiplication binds more tightly:

```text
      +                 not       *
     / \                         / \
    1   *                       +   3
       / \                     / \
      2   3                   1   2
```

Precedence is a property of the grammar and parser—not of the CPU.

```text
expression = term (("+" | "-") term)*
term       = factor (("*" | "/") factor)*
factor     = NUMBER | "(" expression ")"
```

An AST is just a data structure representing program structure. Here, every
node is a tuple: `("number", 5)` or `("binary", "+", left, right)`.


In [10]:
def parse(source):
    tokens = tokenize(source)
    position = 0

    def peek():
        if position < len(tokens):
            return tokens[position]
        return ("EOF", "")

    def consume(expected_kind):
        nonlocal position
        token = peek()
        if token[0] != expected_kind:
            raise ValueError("Expected " + expected_kind + ", got " + token[0])
        position += 1
        return token

    def parse_factor():
        if peek()[0] == "NUMBER":
            return ("number", int(consume("NUMBER")[1]))
        if peek()[0] == "LPAREN":
            consume("LPAREN")
            node = parse_expression()
            consume("RPAREN")
            return node
        raise ValueError("Expected a number or '('")

    def parse_term():
        node = parse_factor()
        while peek()[0] in ("MUL", "DIV"):
            operator = consume(peek()[0])[1]
            node = ("binary", operator, node, parse_factor())
        return node

    def parse_expression():
        node = parse_term()
        while peek()[0] in ("PLUS", "MINUS"):
            operator = consume(peek()[0])[1]
            node = ("binary", operator, node, parse_term())
        return node

    tree = parse_expression()
    if peek()[0] != "EOF":
        raise ValueError("Unexpected token: " + peek()[1])
    return tree


In [11]:
tree_1 = parse("1 + 2 * 3")
tree_2 = parse("(1 + 2) * 3")
print(tree_1)
print(tree_2)

assert tree_1 == (
    "binary", "+", ("number", 1),
    ("binary", "*", ("number", 2), ("number", 3)),
)


('binary', '+', ('number', 1), ('binary', '*', ('number', 2), ('number', 3)))
('binary', '*', ('binary', '+', ('number', 1), ('number', 2)), ('number', 3))


### Parsers and pushdown automata

Lexing connects naturally to regular languages and finite automata. Nested
input such as `(((1 + 2) * 3) + 4)` needs memory for arbitrary nesting, which
connects context-free grammars to stacks and pushdown automata.

```text
parse_factor → parse_expression → parse_factor → ...
```

Our recursive-descent parser uses Python's call stack for this syntactic
memory. Real parsers need not literally be implemented as pushdown automata,
just as real lexers need not instantiate an explicit finite-state-machine
object. These are useful classical models connecting programming languages to
models of computation.


## 4. AST interpreter

An interpreter walks the AST and computes its meaning directly.


In [33]:
def evaluate(node):
    kind = node[0]
    if kind == "number":
        return node[1]
    if kind == "binary":
        operator, left, right = node[1], node[2], node[3]
        a = evaluate(left)
        b = evaluate(right)
        if operator == "+": return a + b
        if operator == "-": return a - b
        if operator == "*": return a * b
        if operator == "/": return a // b
    raise ValueError("Unknown AST node: " + str(node))


In [34]:
assert evaluate(parse("1 + 2 * 3")) == 7
assert evaluate(parse("(1 + 2) * 3")) == 9
assert evaluate(parse("10 / 2 + 3")) == 8
print(evaluate(parse("1 + 2 * 3")))
print(evaluate(parse("(1 + 2) * 3")))
print(evaluate(parse("10 / 2 + 3")))


7
9
8


We have built an interpreter:

```text
source → lexer → tokens → parser → AST → evaluate → result
```


## 5. Compiling AST to bytecode

Instead of executing the AST immediately, we can translate it into
instructions for a tiny stack machine. Each instruction is also a tuple.
`compile_ast` compiles only an expression; it never inserts `PRINT` recursively.


In [13]:
INSTRUCTION_FOR = {"+": "ADD", "-": "SUB", "*": "MUL", "/": "DIV"}

def compile_ast(node):
    kind = node[0]
    if kind == "number":
        return [("PUSH", node[1])]
    if kind == "binary":
        operator, left, right = node[1], node[2], node[3]
        return (compile_ast(left) + compile_ast(right)
                + [(INSTRUCTION_FOR[operator],)])
    raise ValueError("Unknown AST node: " + str(node))

print(*compile_ast(parse("1 + 2 * 3")), sep="\n")


('PUSH', 1)
('PUSH', 2)
('PUSH', 3)
('MUL',)
('ADD',)


An expression and a statement are different layers. For now every statement
is an expression statement, so statement compilation appends exactly one
`PRINT`. This distinction will matter when assignments are added.


In [53]:
def print_bytecode(bytecode):
    # converting to strings for better readability:
    return '\n'.join([' '.join(map(str, instruction)) for instruction in bytecode])

def compile_statement(ast):
    return print_bytecode(compile_ast(ast) + [("PRINT",)])

bytecode = compile_statement(parse("1 + 2 * 3"))
print(bytecode)


PUSH 1
PUSH 2
PUSH 3
MUL
ADD
PRINT


## 6. Stack virtual machine

The VM repeatedly changes machine state: `(stack, environment, instruction)`
becomes a new `(stack, environment)`. `LOAD` and `STORE` are included now for
the homework, although today's lexer, parser, and compiler cannot emit them.


In [65]:
def run(bytecode, env=None):
    if isinstance(bytecode, str):
        # splitting instructions according to semicolons:
        bytecode = bytecode.replace(';', '\n')
        bytecode = bytecode.splitlines()
    stack = []
    if env is None:
        env = {}

    for instruct in bytecode:
        instruction = instruct.split()
        operation = instruction[0]
        if operation == '#':
            continue # comment
        if operation == "PUSH":
            stack.append(int(instruction[1]))
        elif operation in ("ADD", "SUB", "MUL", "DIV", "POW", "MOD"):
            right = stack.pop()
            left = stack.pop()
            if operation == "ADD":
                stack.append(left + right)
            elif operation == "SUB":
                stack.append(left - right)
            elif operation == "MUL":
                stack.append(left * right)
            elif operation == "DIV":
                stack.append(left // right)
            elif operation == "POW":
                stack.append(left ** right)
            elif operation == "MOD":
                stack.append(left % right)
        elif operation == "PRINT":
            print(stack.pop())
        elif operation == "LOAD":
            stack.append(env[instruction[1]])
        elif operation == "STORE":
            env[instruction[1]] = stack.pop()
        else:
            raise ValueError("Unknown instruction: " + operation)
    return env


In [59]:
run(compile_statement(parse("1 + 2 * 3")))

7


{}

In [60]:
print(compile_statement(parse("1 + 2 * 3")))

PUSH 1
PUSH 2
PUSH 3
MUL
ADD
PRINT


This hand-written bytecode corresponds conceptually to `a = 2 + 3` followed
by `a * 4`. The VM can run it, but our source-language frontend cannot produce
it **yet**.


In [62]:
manual_bytecode = 'PUSH 2; PUSH 3; ADD; STORE a; LOAD a; PUSH 4; MUL; PRINT'
manual_env = run(manual_bytecode)
assert manual_env == {"a": 5}


20


## 7. Multiline programs

A program contains one expression per non-blank line. The interpreter prints
each evaluated result. The compiler compiles each line as a statement, so it
automatically adds one `PRINT` per expression.

For simplicity, let's also allow to replace newlines with semicolons: ';'


In [63]:
def program_lines(program_text):
    # if there are semicolons, we replace them with newlines:
    program_text = program_text.replace(';', '\n')
    return [line.strip() for line in program_text.splitlines() if line.strip()]

def interpret(program_text):
    for line in program_lines(program_text):
        print(evaluate(parse(line)))

def compile_program(program_text):
    bytecode = []
    for line in program_lines(program_text):
        bytecode.append('# ' + line)
        bytecode.append(compile_statement(parse(line)))
    return '\n'.join(bytecode)


In [68]:
program = """
1 + 2
3 * 4
(5 + 1) / 2
"""

print("Interpreter:")
interpret(program)
print("VM:")
compiled_program = compile_program(program)
print(compiled_program)
print("Output:")
run(compiled_program)
#assert compiled_program.count(("PRINT",)) == 3


Interpreter:
3
12
3
VM:
# 1 + 2
PUSH 1
PUSH 2
ADD
PRINT
# 3 * 4
PUSH 3
PUSH 4
MUL
PRINT
# (5 + 1) / 2
PUSH 5
PUSH 1
ADD
PUSH 2
DIV
PRINT
Output:
3
12
3


{}

## 8. Putting everything together

Both routes should have identical visible behavior. We capture their output
only to test that claim; the implementation itself still uses ordinary
`print` calls.


In [32]:
import contextlib
import io

def captured_output(action):
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        action()
    return stream.getvalue()

test_programs = [
    "1 + 2 * 3; (1 + 2) * 3; 10 / 2 + 3",
    "1 + 2; 3 * 4; (5 + 1) / 2",
]

for text in test_programs:
    interpreted = captured_output(lambda text=text: interpret(text))
    compiled = captured_output(lambda text=text: run(compile_program(text)))
    assert interpreted == compiled
    print(interpreted, end="")


7
9
8.0
3
12
3.0


## 9. Connection to CPython

CPython follows a recognizable high-level pipeline:

```text
Python source → parser → AST → compiler → bytecode → CPython VM
```

The real system is much richer, and its bytecode details vary between Python
versions. We only inspect the overall shape here.


In [16]:
import dis

def f(a, b, c):
    return a + b * c

dis.dis(f)


  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (a)
              4 LOAD_FAST                1 (b)
              6 LOAD_FAST                2 (c)
              8 BINARY_OP                5 (*)
             12 BINARY_OP                0 (+)
             16 RETURN_VALUE


## 10. Small optional exercises

### Exercise A — modulo

Add `%` with the same precedence as `*` and `/`. Propagate it through lexer,
parser, interpreter, compiler, and VM using the instruction `MOD`.

### Exercise B — exponentiation

Add `**` above multiplication. Make it right-associative, so `2 ** 3 ** 2`
means `2 ** (3 ** 2)`, and use the VM instruction `POW`.

Instructor solutions are in the final removable section.


## 11. Homework: Variables and Assignment

# Homework — Variables and Assignment

Turn the calculator into a tiny stateful language. Multiline execution,
`PRINT`, `LOAD`, `STORE`, the VM environment, arithmetic expressions, and the
expression compiler are already provided.

Implement only these frontend/compiler additions:

1. Lexer tokens `NAME` and `ASSIGN`, where names are exactly `a`–`f`.
2. Variable references and assignment statements in the parser.
3. AST tuples `("variable", name)` and `("assign", name, expression)`.
4. Compiler support for `LOAD` and `STORE`.

```text
statement  = NAME "=" expression | expression
expression = term (("+" | "-") term)*
term       = factor (("*" | "/") factor)*
factor     = NUMBER | NAME | "(" expression ")"
```

Use two-token lookahead: `NAME ASSIGN` begins an assignment. A plain expression
must end in `PRINT`; an assignment must end in `STORE` and print nothing. The
AST interpreter does **not** need homework support for variables.


In [35]:
# TODO 1: Copy and extend TOKEN_PATTERN/tokenize with:
#   (?P<NAME>[a-f])
#   (?P<ASSIGN>=)
#
# TODO 2: Copy and extend parse. A NAME in factor becomes:
#   ("variable", name)
# A leading NAME followed by ASSIGN becomes:
#   ("assign", name, expression_ast)
#
# TODO 3: Extend compile_ast so a variable emits ("LOAD", name).
# Extend compile_statement so assignment emits RHS bytecode + ("STORE", name),
# while an expression still emits expression bytecode + ("PRINT",).


Use these checks after completing the TODOs. They intentionally remain
commented so this starter notebook runs before the homework is implemented.


In [18]:
# assert tokenize("a") == [("NAME", "a")]
# assert tokenize("a = 10") == [("NAME", "a"), ("ASSIGN", "="), ("NUMBER", "10")]
# assert parse("a") == ("variable", "a")
# assert parse("a = 2 + 3") == (
#     "assign", "a", ("binary", "+", ("number", 2), ("number", 3))
# )
# assert compile_ast(("variable", "a")) == [("LOAD", "a")]
# assert compile_statement(parse("a = 2 + 3"))[-1] == ("STORE", "a")
#
# program = "a = 2 + 3\nb = a * 4\nb + 1"
# assert captured_output(lambda: run(compile_program(program))) == "21\n"


Homework examples:

```text
a = 5          a = 2 + 3       2 + 3        a = 2
a + 1          b = a * 4       a = 10       b = 3
                 b + 1          a * 2        a + b
                                 a + 5        a * b

output: 6      output: 21      output:       output:
                                5             5
                                20            6
                                15
```

Assignment lines themselves produce no visible output.


## 13. Summary

```text
Source
  ↓
Lexer          regular languages ↔ finite automata
  ↓
Tokens
  ↓
Parser         context-free grammars ↔ stack / pushdown automata
  ↓
 AST
 /  \
Interpreter  Compiler
               ↓
            Bytecode
               ↓
               VM      machine state → instruction → new machine state
```

Adding a language feature usually propagates through several layers:

```text
a = 2 + 3
  → NAME ASSIGN NUMBER PLUS NUMBER
  → ("assign", ...)
  → PUSH, PUSH, ADD, STORE
  → env["a"] = 5
```

That propagation—from source syntax to machine behavior—is the main takeaway.


## 12. Instructor Solutions — REMOVE BEFORE DISTRIBUTION

# Instructor Solutions
## REMOVE THIS ENTIRE SECTION BEFORE DISTRIBUTING TO STUDENTS

Everything below this heading is a solution. Delete this one contiguous final
section before sharing the student version.


### Homework solution: complete lexer and parser

The solution uses separate `hw_...` names so the seminar implementation above
remains visibly unchanged and arithmetic-only.


In [19]:
HW_TOKEN_PATTERN = re.compile(
    r"(?P<NUMBER>\d+)|(?P<NAME>[a-f])|(?P<ASSIGN>=)"
    r"|(?P<PLUS>\+)|(?P<MINUS>-)|(?P<MUL>\*)|(?P<DIV>/)"
    r"|(?P<LPAREN>\()|(?P<RPAREN>\))"
    r"|(?P<SPACE>\s+)|(?P<MISMATCH>.)"
)

def hw_tokenize(source):
    tokens = []
    for match in HW_TOKEN_PATTERN.finditer(source):
        kind, text = match.lastgroup, match.group()
        if kind == "SPACE":
            continue
        if kind == "MISMATCH":
            raise ValueError("Unexpected character: " + text)
        tokens.append((kind, text))
    return tokens

assert hw_tokenize("a") == [("NAME", "a")]
assert hw_tokenize("a = 10") == [
    ("NAME", "a"), ("ASSIGN", "="), ("NUMBER", "10")
]


In [20]:
def hw_parse(source):
    tokens = hw_tokenize(source)
    position = 0

    def peek(offset=0):
        index = position + offset
        return tokens[index] if index < len(tokens) else ("EOF", "")

    def consume(kind):
        nonlocal position
        token = peek()
        if token[0] != kind:
            raise ValueError("Expected " + kind + ", got " + token[0])
        position += 1
        return token

    def factor():
        if peek()[0] == "NUMBER":
            return ("number", int(consume("NUMBER")[1]))
        if peek()[0] == "NAME":
            return ("variable", consume("NAME")[1])
        if peek()[0] == "LPAREN":
            consume("LPAREN")
            node = expression()
            consume("RPAREN")
            return node
        raise ValueError("Expected a number, variable, or '('")

    def term():
        node = factor()
        while peek()[0] in ("MUL", "DIV"):
            operator = consume(peek()[0])[1]
            node = ("binary", operator, node, factor())
        return node

    def expression():
        node = term()
        while peek()[0] in ("PLUS", "MINUS"):
            operator = consume(peek()[0])[1]
            node = ("binary", operator, node, term())
        return node

    if peek()[0] == "NAME" and peek(1)[0] == "ASSIGN":
        name = consume("NAME")[1]
        consume("ASSIGN")
        tree = ("assign", name, expression())
    else:
        tree = expression()
    if peek()[0] != "EOF":
        raise ValueError("Unexpected token: " + peek()[1])
    return tree


In [21]:
assert hw_parse("a") == ("variable", "a")
assert hw_parse("a = 2 + 3") == (
    "assign", "a", ("binary", "+", ("number", 2), ("number", 3))
)
print(hw_parse("a = 2 + 3"))


('assign', 'a', ('binary', '+', ('number', 2), ('number', 3)))


### Homework solution: compiler and complete pipeline

Variable expressions emit `LOAD`. Assignment statements compile their right
side and emit `STORE` without `PRINT`; expression statements still print.


In [22]:
def hw_compile_ast(node):
    kind = node[0]
    if kind == "number":
        return [("PUSH", node[1])]
    if kind == "variable":
        return [("LOAD", node[1])]
    if kind == "binary":
        operator, left, right = node[1], node[2], node[3]
        return (hw_compile_ast(left) + hw_compile_ast(right)
                + [(INSTRUCTION_FOR[operator],)])
    raise ValueError("Expected an expression AST")

def hw_compile_statement(node):
    if node[0] == "assign":
        return hw_compile_ast(node[2]) + [("STORE", node[1])]
    return hw_compile_ast(node) + [("PRINT",)]

def hw_compile_program(program_text):
    bytecode = []
    for line in program_lines(program_text):
        bytecode += hw_compile_statement(hw_parse(line))
    return bytecode

assert hw_compile_ast(("variable", "a")) == [("LOAD", "a")]
assert hw_compile_statement(hw_parse("a = 2 + 3")) == [
    ("PUSH", 2), ("PUSH", 3), ("ADD",), ("STORE", "a")
]


In [23]:
homework_program = """
a = 2 + 3
b = a * 4
b + 1
"""
homework_output = captured_output(lambda: run(hw_compile_program(homework_program)))
assert homework_output == "21\n"
print(homework_output, end="")

mixed_program = """
2 + 3
a = 10
a * 2
"""
mixed_output = captured_output(lambda: run(hw_compile_program(mixed_program)))
assert mixed_output == "5\n20\n"
print(mixed_output, end="")


21
5
20


In [24]:
final_program = """
a = 10
b = a + 5
b * 2
a + b
"""
final_bytecode = hw_compile_program(final_program)
print(*final_bytecode, sep="\n")
print("Output:")
final_env = run(final_bytecode)
assert final_env == {"a": 10, "b": 15}


('PUSH', 10)
('STORE', 'a')
('LOAD', 'a')
('PUSH', 5)
('ADD',)
('STORE', 'b')
('LOAD', 'b')
('PUSH', 2)
('MUL',)
('PRINT',)
('LOAD', 'a')
('LOAD', 'b')
('ADD',)
('PRINT',)
Output:
30
25


### Optional exercise solutions

For modulo, add `(?P<MOD>%)`, accept `MOD` beside `MUL`/`DIV`, evaluate `%`,
map `%` to `MOD`, and add a VM arithmetic branch using `left % right`.

For exponentiation, put `(?P<POW>\*\*)` **before** the single-star pattern.
Add a grammar level such as:

```text
power = factor ("**" power)?
term  = power (("*" | "/" | "%") power)*
```

The recursive call on the right makes exponentiation right-associative. Add
evaluation with `**`, map it to `POW`, and implement `left ** right` in the VM.


In [25]:
# Compact checks of the two new arithmetic operations used by those solutions.
def optional_vm_operation(operation, left, right):
    if operation == "MOD":
        return left % right
    if operation == "POW":
        return left ** right
    raise ValueError("Unknown optional operation")

assert optional_vm_operation("MOD", 17, 5) == 2
assert optional_vm_operation("POW", 2, 3 ** 2) == 512
print("Optional operation checks passed.")


Optional operation checks passed.
